# Lesson 03: 3D 시각화

## 학습 목표
1. Irrlicht/VSG 시각화 시스템 설정하기
2. 물체에 시각적 형태(Visual Shape) 입히기
3. 카메라, 조명, 하늘 배경 설정하기
4. 실시간 시뮬레이션 루프 실행하기

## 새로 배우는 Chrono API
| 클래스/메서드 | 역할 |
|---|---|
| `ChVisualSystemIrrlicht()` | Irrlicht 3D 렌더러 |
| `ChVisualSystemVSG()` | Vulkan 기반 렌더러 (macOS 권장) |
| `vis.AttachSystem(sys)` | 물리 시스템과 렌더러 연결 |
| `ChVisualShapeBox/Sphere(...)` | 눈에 보이는 형태 |
| `ChColor(r, g, b)` | 색상 (0.0~1.0) |
| `AddVisualShape(shape)` | 물체에 시각 형태 추가 |
| `ChRealtimeStepTimer` | 실시간 동기화 타이머 |

> **참고**: 시각화는 Jupyter에서 실행 불가. 터미널에서 `python lessons/phase1/lesson_03_visualization.py` 실행.
> 이 노트북에서는 코드 구조와 API를 이해하는 데 집중합니다.

## 핵심 개념: 충돌 형태 vs 시각 형태

Chrono에서 물체의 "모양"은 두 가지가 분리되어 있습니다:

| | 충돌 형태 (Collision Shape) | 시각 형태 (Visual Shape) |
|---|---|---|
| 역할 | 물리 엔진이 충돌 감지에 사용 | 렌더러가 화면에 그리는 데 사용 |
| 추가 방법 | `AddCollisionShape(...)` | `AddVisualShape(...)` |
| 필수 여부 | 충돌이 필요한 경우만 | 시각화가 필요한 경우만 |
| 크기 | 동일하게 맞추는 것이 자연스러움 | 독립적으로 설정 가능 |

> Lesson 02에서는 충돌 형태만 있었고, 여기서 시각 형태를 추가합니다.

In [ ]:
import pychrono as chrono

# 시스템 + 충돌 설정 (Lesson 02 복습)
sys = chrono.ChSystemNSC()
sys.SetGravitationalAcceleration(chrono.ChVector3d(0, -9.81, 0))
sys.SetCollisionSystemType(chrono.ChCollisionSystem.Type_BULLET)

material = chrono.ChContactMaterialNSC()
material.SetFriction(0.4)
material.SetRestitution(0.5)

## 바닥: 충돌 형태 + 시각 형태 함께 추가

In [ ]:
floor = chrono.ChBody()
floor.SetPos(chrono.ChVector3d(0, -1, 0))
floor.SetFixed(True)
floor.EnableCollision(True)
floor.AddCollisionShape(chrono.ChCollisionShapeBox(material, 20, 2, 20))

# ★ 시각 형태 추가 (크기를 충돌 형태와 동일하게)
floor_shape = chrono.ChVisualShapeBox(20, 2, 20)
floor_shape.SetColor(chrono.ChColor(0.4, 0.5, 0.4))  # 연한 녹색
floor.AddVisualShape(floor_shape)

sys.AddBody(floor)
print("바닥 생성 (충돌 + 시각 형태)")

## 공 생성 함수화

여러 물체를 만들 때 함수로 묶으면 편리합니다.

In [ ]:
def create_ball(system, mat, pos, radius, color, name):
    """충돌 + 시각 형태를 가진 공 생성"""
    ball = chrono.ChBody()
    ball.SetMass(1.0)
    ball.SetPos(pos)
    ball.SetFixed(False)
    ball.EnableCollision(True)

    # 충돌 형태
    ball.AddCollisionShape(chrono.ChCollisionShapeSphere(mat, radius))

    # 관성 모멘트 (구: I = 2/5 * m * r²)
    inertia = 2.0 / 5.0 * ball.GetMass() * radius**2
    ball.SetInertiaXX(chrono.ChVector3d(inertia, inertia, inertia))

    # ★ 시각 형태
    ball_shape = chrono.ChVisualShapeSphere(radius)
    ball_shape.SetColor(color)
    ball.AddVisualShape(ball_shape)

    system.AddBody(ball)
    print(f"  {name}: pos=({pos.x}, {pos.y}, {pos.z}), r={radius}")
    return ball

# 공 3개 생성
print("공 생성:")
ball_red = create_ball(sys, material, chrono.ChVector3d(-2, 8, 0), 0.4,
                       chrono.ChColor(0.9, 0.2, 0.2), "빨간 공")
ball_blue = create_ball(sys, material, chrono.ChVector3d(0, 5, 0), 0.3,
                        chrono.ChColor(0.2, 0.3, 0.9), "파란 공")
ball_yellow = create_ball(sys, material, chrono.ChVector3d(2, 3, 0), 0.35,
                          chrono.ChColor(0.9, 0.8, 0.1), "노란 공")

## 시각화 시스템 설정 (VSG/Irrlicht 자동 분기)

아래는 터미널에서 실행할 때의 시각화 코드 구조입니다:

```python
# VSG 우선, Irrlicht 폴백
try:
    import pychrono.vsg3d as chronovsg
    USE_VSG = True
except ImportError:
    USE_VSG = False
import pychrono.irrlicht as chronoirr

if USE_VSG:
    vis = chronovsg.ChVisualSystemVSG()
    vis.SetCameraVertical(chrono.CameraVerticalDir_Y)  # VSG는 Z-up 기본이므로 Y-up 설정
else:
    vis = chronoirr.ChVisualSystemIrrlicht()

vis.AttachSystem(sys)
vis.SetWindowSize(1280, 720)
vis.SetWindowTitle('Lesson 03: Bouncing Balls')

# ★ VSG vs Irrlicht 초기화 순서가 다름!
if USE_VSG:
    vis.AddCamera(...)   # VSG: 카메라 먼저
    vis.Initialize()     # 그 다음 초기화
else:
    vis.Initialize()     # Irrlicht: 초기화 먼저
    vis.AddSkyBox()      # 하늘 배경
    vis.AddCamera(...)   # 그 다음 카메라
    vis.AddTypicalLights()  # 조명
```

### 시뮬레이션 루프
```python
realtime_timer = chrono.ChRealtimeStepTimer()  # macOS 필수

while vis.Run():           # 창이 열려있는 동안
    vis.BeginScene()       # 프레임 시작
    vis.Render()           # 3D 장면 그리기
    vis.EndScene()         # 프레임 끝
    sys.DoStepDynamics(0.005)
    realtime_timer.Spin(0.005)  # 실시간 동기화
```

## 시각화 없이 시뮬레이션 실행 (노트북용)

시각화는 터미널에서만 가능하지만, 물리 시뮬레이션 자체는 노트북에서도 실행 가능합니다.

In [ ]:
# 3초간 시뮬레이션 (시각화 없이)
while sys.GetChTime() < 3.0:
    sys.DoStepDynamics(0.005)

print(f"시뮬레이션 {sys.GetChTime():.2f}초 완료")
print(f"빨간 공: y = {ball_red.GetPos().y:.3f} m")
print(f"파란 공: y = {ball_blue.GetPos().y:.3f} m")
print(f"노란 공: y = {ball_yellow.GetPos().y:.3f} m")

## 정리

| 개념 | 설명 |
|---|---|
| 충돌 형태 ≠ 시각 형태 | 둘 다 따로 추가해야 함 |
| VSG vs Irrlicht | 초기화 순서가 다름 (VSG: 카메라→Init, Irrlicht: Init→카메라) |
| `ChRealtimeStepTimer` | macOS에서 vsync 미지원 → 이 타이머로 실시간 동기화 |
| `ChColor(r, g, b)` | 0.0~1.0 범위의 RGB 색상 |